<a href="https://colab.research.google.com/github/ddAdoro/AI-Assignment-SCT314-C004-0594-2026/blob/main/Notebooks/Chap13/13_4_Graph_Attention_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.4: Graph attention networks**

This notebook builds a graph attention mechanism from scratch, as discussed in section 13.8.6 of the book and illustrated in figure 13.12c

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [1]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [2]:
# Set seed so we get the same random numbers
np.random.seed(1)
# Number of nodes in the graph
N = 8
# Number of dimensions of each input
D = 4

# Define a graph
A = np.array([[0,1,0,1,0,0,0,0],
              [1,0,1,1,1,0,0,0],
              [0,1,0,0,1,0,0,0],
              [1,1,0,0,1,0,0,0],
              [0,1,1,1,0,1,0,1],
              [0,0,0,0,1,0,1,1],
              [0,0,0,0,0,1,0,0],
              [0,0,0,0,1,1,0,0]]);
print(A)

# Let's also define some random data
X = np.random.normal(size=(D,N))

[[0 1 0 1 0 0 0 0]
 [1 0 1 1 1 0 0 0]
 [0 1 0 0 1 0 0 0]
 [1 1 0 0 1 0 0 0]
 [0 1 1 1 0 1 0 1]
 [0 0 0 0 1 0 1 1]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 1 1 0 0]]


We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [3]:
# Choose random values for the parameters
omega = np.random.normal(size=(D,D))
beta = np.random.normal(size=(D,1))
phi = np.random.normal(size=(2*D,1))

We'll need a softmax operation that operates on the columns of the matrix and a ReLU function as well

In [4]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in) ;
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax


# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation


In [8]:
 # Now let's compute self attention in matrix form
def graph_attention(X,omega, beta, phi, A):

  # D and N are defined in the global scope but also can be derived from X
  N = X.shape[1] # Number of nodes
  D = X.shape[0] # Dimension of each input

  # 1. Compute X_prime
  X_prime = omega @ X

  # 2. Compute S_nm = phi_1.T @ X'_n + phi_2.T @ X'_m
  # Split phi into two D x 1 vectors (phi is 2D x 1)
  phi_1 = phi[:D, :]
  phi_2 = phi[D:, :]

  # Calculate the two terms for S:
  # term_n is 1 x N where term_n[0, n] = phi_1.T @ X_prime[:, n]
  term_n = phi_1.T @ X_prime
  # term_m is 1 x N where term_m[0, m] = phi_2.T @ X_prime[:, m]
  term_m = phi_2.T @ X_prime

  # Construct S (N x N) where S[n, m] = term_n.T[n, 0] + term_m[0, m]
  # Broadcasting (N, 1) + (1, N) results in (N, N)
  S = term_n.T + term_m

  # 3. To apply the mask, set S to a very large negative number (e.g. -1e20) everywhere where A+I is zero
  A_plus_I = A + np.eye(N) # Add identity matrix for self-loops
  mask = (A_plus_I == 0)
  S[mask] = -1e20 # Mask non-neighboring nodes

  # 4. Run the softmax function to compute the attention values
  # The provided softmax_cols function normalizes columns.
  # For graph attention (a_nm, where sum_m a_nm = 1 for a given n), we need row-wise softmax.
  # This is achieved by transposing S, applying softmax_cols, then transposing back.
  attention = softmax_cols(S.T).T

  # 5. Postmultiply X' by the attention values
  # The output X_prime_n should be a weighted sum of neighbor's X_prime_m:
  # output_prime[:, n] = sum_m (attention[n, m] * X_prime[:, m])
  # This corresponds to the matrix multiplication X_prime @ attention.T
  output_prime = X_prime @ attention.T

  # 6. Apply the ReLU function with bias
  output = ReLU(output_prime + beta)

  return output;

In [6]:
# Test out the graph attention mechanism
np.set_printoptions(precision=3)
output = graph_attention(X, omega, beta, phi, A);
print("Correct answer is:")
print("[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]")
print(" [0.    0.    0.    0.    1.184 0.    2.654 0.  ]")
print(" [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]")
print(" [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]")


print("Your answer is:")
print(output)

Correct answer is:
[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]
 [0.    0.    0.    0.    1.184 0.    2.654 0.  ]
 [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]
 [0.825 0.    0.    1.175 0.    0.    0.    0.  ]]]
Your answer is:
[[1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1. 1. 1. 1.]]


TODO -- Try to construct a dot-product self-attention mechanism as in practical 12.1 that respects the geometry of the graph and has zero attention between non-neighboring nodes by combining figures 13.12a and 13.12b.
